In [2]:
import os
print("Notebook working directory:", os.getcwd())

Notebook working directory: /Users/emmaphan/Documents/Comp Chem/Cl Transporter/script


In [3]:
import sys
import os

# Project root
project_root = "/Users/emmaphan/Documents/Comp Chem/Cl Transporter"
sys.path.append(os.path.join(project_root, "models"))
sys.path.append(project_root)

print("Paths added:", sys.path[-2:])

Paths added: ['/Users/emmaphan/Documents/Comp Chem/Cl Transporter/models', '/Users/emmaphan/Documents/Comp Chem/Cl Transporter']


In [4]:
import models.fm4m as fm4m
import pandas as pd
import numpy as np

import pyarrow as pa
import pyarrow.parquet as pq


from sklearn.svm import SVR
from sklearn.compose import TransformedTargetRegressor
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_iris

from rdkit import Chem
from rdkit.Chem.Descriptors import CalcMolDescriptors
from rdkit.Chem import MolFromSmiles

print ("okay")

okay


In [5]:
train_df = pd.read_csv("/Users/emmaphan/Documents/Comp Chem/Cl Transporter/data/training.csv")
test_df = pd.read_csv("/Users/emmaphan/Documents/Comp Chem/Cl Transporter/data/test.csv")

print (train_df.shape, test_df.shape)

(1348, 7) (175, 3)


In [6]:
#train_smiles = train_df["SMILES"].to_list()
test_smiles = test_df["SMILES"].to_list()
test_activity = test_df["Activity (1 = active, 0 = inactive)"].to_list()
len(test_smiles)
print (test_activity)

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [7]:
#multiple undersampling

train_active = train_df[train_df['Activity (1 = active, 0 = inactive)'] == 1]
train_inactive = train_df[train_df['Activity (1 = active, 0 = inactive)'] == 0]

print(len(train_active))    # =54
print(len(train_inactive))  # =1294

inactive_shuffled = train_inactive.sample(frac=1, random_state=42)
inactive_chunks = np.array_split(inactive_shuffled, 19)
print([len(chunk) for chunk in inactive_chunks])

training_sets = []

for i, inactive_chunk in enumerate(inactive_chunks):
    subset = pd.concat(
        [train_active, inactive_chunk],
        ignore_index=True
    )
    training_sets.append(subset)

train_smiles = {}
train_activity = {} 

for i, col in enumerate(training_sets):
    train_smiles[i] = col["SMILES"].to_list()
    train_activity[i] = col["Activity (1 = active, 0 = inactive)"].to_list()

len(train_smiles[1])

54
1294
[69, 69, 68, 68, 68, 68, 68, 68, 68, 68, 68, 68, 68, 68, 68, 68, 68, 68, 68]


/opt/anaconda3/envs/smi-ted-env/lib/python3.9/site-packages/numpy/core/fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


123

In [8]:
print(train_activity[1])

[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [9]:
#SMILES to descriptor
from rdkit import Chem
from rdkit.Chem.Descriptors import CalcMolDescriptors
from rdkit.Chem import MolFromSmiles

def features(smiles):
      m = MolFromSmiles(smiles)
      vals = list(CalcMolDescriptors(m).values())
      return vals
    

train_rdkit = {}  # new dictionary

for i in train_smiles:
    rdkit_list = []  # temporary list for this key
    for mol in train_smiles[i]:
        rdkit_value = features(mol)  # translate each element
        rdkit_list.append(rdkit_value)  # add to the temp list
        train_rdkit[i] = rdkit_list  # assign the translated list to the new dict
    
test_rdkit = []
for smiles in test_smiles: 
    test_rdkit.append(features(smiles))


In [10]:
print (len(train_smiles[1]))
print (len(train_rdkit[1]))
len(train_activity[1])
#print (train_rdkit[1])

123
123


123

In [11]:
import sklearn
print(sklearn.__version__)

1.6.1


In [12]:
#RF
rf5 = RandomForestClassifier(n_estimators=5, max_depth = 10, random_state=42)
rf5.fit (train_rdkit[1],train_activity[1])
rf5_pred = rf5.predict(test_rdkit)
print("okay")

print (rf5_pred)
accuracy = accuracy_score(test_activity, rf5_pred)
print (accuracy)


okay
[0 1 0 0 1 0 0 0 0 0 0 0 0 0 1 1 1 0 0 0 0 0 0 1 0 0 0 0 0 0 1 0 0 1 0 0 0
 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 1 1 1 1 0 0 0 1 1 1 0 1 0 0 0
 0 0 0 0 0 0 0 0 1 0 0 0 0 1 0 0 0 0 0 0 0 0 0 1 1 1 0 0 0 0 0 1 0 0 0 0 1
 0 1 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0
 0 0 1 1 1 1 1 1 1 0 1 1 1 1 1 1 1 0 0 0 1 1 1 0 1 0 1]
0.8171428571428572


In [13]:
#try for 1 data set
#model_type = "SMI-TED"
#train_emb, test_emb = fm4m.get_representation(train_smiles[1],test_smiles, model_type, return_tensor=False)

#try on all data set
model_type = "SMI-TED"

train_emb = {}
test_emb = {}

for i in train_smiles:
    train_emb[i], test_emb[i] = fm4m.get_representation(
        train_smiles[i],
        test_smiles,
        model_type,
        return_tensor=True
    )

Random Seed: 12345
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding


/opt/anaconda3/envs/smi-ted-env/lib/python3.9/site-packages/numpy/core/fromnumeric.py:59: FutureWarning: 'Series.swapaxes' is deprecated and will be removed in a future version. Please use 'Series.transpose' instead.
  return bound(*args, **kwds)


Vocab size: 2393
[INFERENCE MODE - smi-ted-Light]


100%|█████████████████████████████████████████████| 1/1 [00:28<00:00, 28.50s/it]


Random Seed: 12345
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Vocab size: 2393
[INFERENCE MODE - smi-ted-Light]


...

100%|█████████████████████████████████████████████| 1/1 [00:32<00:00, 32.10s/it]


Random Seed: 12345
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Vocab size: 2393
[INFERENCE MODE - smi-ted-Light]


100%|█████████████████████████████████████████████| 1/1 [00:28<00:00, 28.79s/it]


Random Seed: 12345
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Using Rotation Embedding
Vocab size: 2393
[INFERENCE MODE - smi-ted-Light]


100%|█████████████████████████████████████████████| 1/1 [00:32<00:00, 32.65s/it]


In [14]:
print(train_emb)

{0: tensor([[ 0.3457, -0.4892,  0.1002,  ...,  0.8100,  0.6090, -0.0267],
        [ 0.4098, -0.4338,  0.0565,  ...,  0.8571,  0.6816, -0.1899],
        [ 0.4631, -0.5216,  0.0923,  ...,  0.8149,  0.6306, -0.1073],
        ...,
        [ 0.5361, -0.5094,  0.0441,  ...,  0.7537,  0.5697, -0.1126],
        [ 0.4179, -0.5579,  0.0471,  ...,  0.9715,  0.6433, -0.2357],
        [ 0.3794, -0.5162,  0.0555,  ...,  0.8102,  0.5745, -0.1365]]), 1: tensor([[ 0.3457, -0.4892,  0.1002,  ...,  0.8100,  0.6090, -0.0267],
        [ 0.4098, -0.4338,  0.0565,  ...,  0.8571,  0.6816, -0.1899],
        [ 0.4631, -0.5216,  0.0923,  ...,  0.8149,  0.6306, -0.1073],
        ...,
        [ 0.4093, -0.5356,  0.0826,  ...,  0.8513,  0.6102, -0.1370],
        [ 0.3401, -0.5559,  0.0515,  ...,  0.8256,  0.6066, -0.1034],
        [ 0.4252, -0.6105,  0.0780,  ...,  0.8691,  0.6292, -0.2355]]), 2: tensor([[ 0.3457, -0.4892,  0.1002,  ...,  0.8100,  0.6090, -0.0267],
        [ 0.4098, -0.4338,  0.0565,  ...,  0.8571,

In [15]:
print(train_emb.shape)

AttributeError: 'dict' object has no attribute 'shape'

In [16]:
type(train_emb)

dict

In [17]:
train_emb[0:2]

TypeError: unhashable type: 'slice'

In [18]:
#RF
rf5 = RandomForestClassifier(n_estimators=15, max_depth = 10, random_state=42)
rf5.fit (train_rdkit[1],train_activity[1])
rf5_pred = rf5.predict(test_rdkit)
print("okay")

print (rf5_pred)
accuracyrf5 = accuracy_score(test_activity, rf5_pred)
print (accuracyrf5)


okay
[0 0 0 0 1 0 0 0 1 0 0 0 0 0 1 1 1 0 0 1 0 0 1 1 0 0 0 0 0 0 1 0 0 1 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 1 0 0 0 0 0 1 1 1 1 0 0 0 1 1 1 0 1 0 0 1
 0 0 1 0 0 0 1 0 1 0 0 0 0 1 0 0 0 0 1 0 0 0 0 1 1 1 0 0 0 0 0 1 0 0 0 0 0
 0 0 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 1 1 0 0
 0 0 1 1 1 1 1 1 1 0 1 1 0 1 1 1 1 0 0 0 1 1 1 0 1 1 0]
0.7771428571428571


In [19]:
rf5 = RandomForestClassifier(n_estimators=15, max_depth = 10, random_state=42)
smited=rf5.fit (train_emb[1],train_activity[1])
smited_pred = rf5.predict(test_emb[1])
print("okay")

print (smited_pred)
accuracy = accuracy_score(test_activity, smited_pred)
print (accuracy)


okay
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 1 0 1 0 1 0 0 0 0 0 0 0 1 0 1 1 0 1
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 1 0 0 0 0 0 0
 0 1 1 0 0 0 1 0 1 0 0 0 0 1 0 0 0 0 1 0 0 0 0 1 0 0 1 0 0 0 0 1 0 0 0 0 1
 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0
 0 0 0 1 1 0 0 0 1 1 0 0 1 1 1 1 0 0 0 0 1 1 1 0 0 0 1]
0.8


In [20]:
#train_smiles[1]

In [21]:
print(len(train_smiles[1]))
print(len(list(train_emb[1])))
print(type(train_emb[1]))
print (len(train_activity[1]))
#all have 123 entries
len(list(train_emb[1]))

train_emb[1].shape

123
123
<class 'torch.Tensor'>
123


torch.Size([123, 768])

In [22]:
#rf5.fit (train_rdkit[1],train_activity[1])
#rf5_pred = rf5.predict(test_rdkit)
print(len(train_rdkit[1]))
print(len(train_activity[1]))
print(len(test_rdkit))

123
123
175


In [23]:
#create Parquet file
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

trainsmiles_col = train_smiles[1]
trainsmited_col = list(train_emb[1])
trainactivity_col = train_activity[1]

frame = {
    "Train SMILES": trainsmiles_col,
    "Train SMILES-TED": [t.detach().cpu().numpy() for t in trainsmited_col],#covert tensor to 
    "Activity": trainactivity_col
}
df = pd.DataFrame(frame)
#df.to_parquet("training_1.parquet", engine="pyarrow")
df


,Train SMILES,Train SMILES-TED,Activity
0,CC(CC(C#N)(C1=CC=CC=C1)C2=CC=CC=C2)N3CCCCC3,"[0.34571952, -0.4891502, 0.10022025, 0.3237361...",1
1,C1=CC(=CC=C1C2=NC3=C(N=C(N=C3N=C2C4=CC=C(C=C4)...,"[0.4098275, -0.4337771, 0.056540385, 0.3883391...",1
2,CCN(CC)CCN1C2=CC=CC=C2SC3=C1C4=CC=CC=C4C=C3,"[0.46312547, -0.5215707, 0.092277415, 0.384997...",1
3,CC(C)(C)NC(=S)NC(C)(C)C,"[0.30377987, -0.49192154, 0.12796305, 0.386750...",1
4,CC1=COC(CC1)(C)C=NCC(=C)C,"[0.42571047, -0.51897144, 0.092369385, 0.43873...",1
...,...,...,...
118,C1=CC=C(C=C1)C(=O)N2C(C=CC3=C2C=CC4=CC=CC=C43)C#N,"[0.35004827, -0.55080354, 0.06762384, 0.358878...",0
119,CC1=C(C=CC(=N1)C(=O)O)OC,"[0.36783245, -0.5063963, 0.08102232, 0.4008098...",0
120,CC(C)C(=O)C1=CC(=C(C=C1)O)O,"[0.40933627, -0.5356133, 0.082636885, 0.381247...",0
121,C1=CC=C(C=C1)C2=CC(=C(N2)C3=NC=CC4=CC=CC=C43)C#N,"[0.34010127, -0.5558511, 0.05154382, 0.3368733...",0


In [24]:
outputtrain = '/Users/emmaphan/Documents/Comp Chem/Cl Transporter/data/training_complete'

for i in range(18):
    trainsmiles_col = train_smiles[i]
    trainsmited_col = list(train_emb[i])
    trainactivity_col = train_activity[i]

    frame = {
        "Train SMILES": trainsmiles_col,
        "Train SMILES-TED": [t.detach().cpu().numpy() for t in trainsmited_col],
        "Activity": trainactivity_col
    }

    df = pd.DataFrame(frame)
    df.to_parquet(
        f"{outputtrain}/training_{i+1}.parquet",
        engine="pyarrow"
    )

In [25]:
print(len(test_smiles))
print(len((test_emb[1])))
print(type(test_emb[1]))
print (len(test_activity))
#all have 123 entries
len(list(train_emb[1]))

test_emb[1].shape
print (test_activity)

175
175
<class 'torch.Tensor'>
175
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [26]:
testsmiles_col = test_smiles
testsmited_col = list(test_emb[1])
testactivity_col = test_activity

frame = {
    "Test SMILES": testsmiles_col,
    "Test SMILES-TED": [t.detach().cpu().numpy() for t in testsmited_col],#covert tensor to 
    "Activity": testactivity_col 
}
df = pd.DataFrame(frame)
#df.to_parquet("training_1.parquet", engine="pyarrow")
df

,Test SMILES,Test SMILES-TED,Activity
0,CCOC(=O)NC1=CC2=C(C=C1)N=C(N2)C3=CSC=N3,"[0.43053472, -0.5042212, 0.021477599, 0.428175...",0
1,CSC1=NC(=C(S1)C(=O)NC2=CC=C(C=C2)[N+](=O)[O-])N,"[0.41468257, -0.5559679, 0.081442624, 0.382502...",0
2,CCCC(=O)N(C)C1=NC2=C(C=C(C=C2)Cl)C(=[N+](C1)[O...,"[0.44089022, -0.5182809, 0.10519825, 0.3779951...",0
3,CC(=O)NC1CCC2=CC(=C(C(=C2C3=CC=C(C(=O)C=C13)SC...,"[0.55152303, -0.4284187, 0.057089463, 0.394849...",0
4,CC1=NC2=C(C=CN2)C(=N1)NCC3=CC=CC=C3,"[0.3535877, -0.5111972, 0.07820781, 0.40937755...",0
...,...,...,...
170,OC1=C(C(NC2=CC(C(F)(F)F)=CC(C(F)(F)F)=C2)=O)C=...,"[0.40523592, -0.42657423, 0.1257377, 0.3822839...",1
171,O=C(/C=C/C(NC1CCCCC1)=O)NC2CCCCC2,"[0.41647348, -0.48067212, 0.09011023, 0.361469...",1
172,O=C(N(CCNC(C1=CC=C(Cl)C=C1)=O)C(N2CCNC(C3=CC=C...,"[0.43154377, -0.42166585, 0.079758115, 0.33677...",1
173,C1(N2CC(CN(CC3=CC=CC=C3)C4)CC4C2)=NC(N5CC6CC(C...,"[0.34182513, -0.4397151, 0.09460954, 0.3177235...",1


In [27]:
outputtest = '/Users/emmaphan/Documents/Comp Chem/Cl Transporter/data/test_complete'

for i in range(18):
    testsmiles_col = test_smiles
    testsmited_col = list(test_emb[1])
    testactivity_col = test_activity

    frame = {
    "Test SMILES": testsmiles_col,
    "Test SMILES-TED": [t.detach().cpu().numpy() for t in testsmited_col],#covert tensor to 
    "Activity": testactivity_col }

    df = pd.DataFrame(frame)
    df.to_parquet(
        f"{outputtest}/test_{i+1}.parquet",
        engine="pyarrow"
    )


In [28]:
#accuracysmited5 = []
for i in range (19): 
    rf5 = RandomForestClassifier(n_estimators=5, max_depth = 10, random_state=42)
    smited=rf5.fit (train_emb[i],train_activity[i])
    smited_pred = rf5.predict(test_emb[i])

    accuracy = accuracy_score(test_activity, smited_pred)
    #accuracysmited5.append(accuracy)
    print(i,accuracy)
    
#print(accuracysmited5)

0 0.7371428571428571
1 0.7714285714285715
2 0.72
3 0.76
4 0.7828571428571428
5 0.6742857142857143
6 0.7257142857142858
7 0.64
8 0.7885714285714286
9 0.7485714285714286
10 0.8171428571428572
11 0.6742857142857143
12 0.7428571428571429
13 0.7657142857142857
14 0.6457142857142857
15 0.7542857142857143
16 0.7371428571428571
17 0.7942857142857143
18 0.7314285714285714


In [29]:
def rf(nestimator, traindata, testdata, accuracylist): 
    total_sum=0
    for i in range (19): 
        rf = RandomForestClassifier(n_estimators=nestimator, max_depth = 10, random_state=42)
        smited=rf.fit (traindata[i],train_activity[i])
        smited_pred = rf.predict(testdata)

        accuracy = accuracy_score(test_activity, smited_pred)
        accuracylist.append(accuracy)
        total_sum += accuracy
        
    average_accuracy = total_sum /len(traindata)
    return print(average_accuracy)
        
accuracysmited5 = []
accuracysmited10 = []
accuracysmited15= []

rf(5,train_emb,test_emb[0],accuracysmited5)
rf(10,train_emb,test_emb[0],accuracysmited10)
rf(15,train_emb,test_emb[0],accuracysmited15)

0.7374436090225563
0.8063157894736844
0.7903759398496241


In [30]:
accuracyrdkit5 = []
accuracyrdkit10 = []
accuracyrdkit15= []

rf(5,train_rdkit,test_rdkit,accuracyrdkit5)
rf(10,train_rdkit,test_rdkit,accuracyrdkit10)
rf(15,train_rdkit,test_rdkit,accuracyrdkit15)

0.7726315789473684
0.8126315789473686
0.7969924812030076


In [34]:
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

clf = make_pipeline(
    StandardScaler(),
    MLPClassifier(random_state=1, max_iter=1000)
)

clf.fit(train_emb[1], train_activity[1])

accuracy = clf.score(test_emb[1], test_activity)
print(accuracy)

0.8285714285714286


In [37]:
X_train_full = np.array(train_rdkit[1])
X_test_full = np.array(test_rdkit)

mask = ~np.isnan(X_train_full).any(axis=0)

X_train = X_train_full[:, mask]
X_test = X_test_full[:, mask]

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier

clf = make_pipeline(
    StandardScaler(),
    MLPClassifier(random_state=1, max_iter=1000)
)

clf.fit(X_train, train_activity[1])

accuracy = clf.score(X_test, test_activity)
print(accuracy)

0.76
